# 01 · Tiny Shakespeare baseline

目标：验证 `数据 → GPT forward → loss → backward → optimizer → checkpoint → sampling` 完整链路。

实验顺序：

1. 确认 Colab GPU 与远端工作目录。
2. 在固定 batch 上过拟合，排除训练链路错误。
3. 运行 baseline 并观察 train/val loss。
4. 使用固定 prompt 采样。
5. 重新加载 checkpoint，确认产物可恢复。

In [ ]:
import json
import os
import platform
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

print('python:', platform.python_version())
print('torch:', torch.__version__)
print('cwd:', Path.cwd())
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    subprocess.run(['nvidia-smi'], check=False)

In [ ]:
def find_stage_dir(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if candidate.name == '03-nanogpt' and (candidate / 'model.py').exists():
            return candidate
        stage_dir = candidate / '03-nanogpt'
        if (stage_dir / 'model.py').exists():
            return stage_dir
    raise FileNotFoundError('找不到 03-nanogpt；请确认 VS Code Colab runtime 能看到本地工作区文件')

STAGE_DIR = find_stage_dir(Path.cwd().resolve())
if str(STAGE_DIR) not in sys.path:
    sys.path.insert(0, str(STAGE_DIR))
os.chdir(STAGE_DIR)
print('stage dir:', STAGE_DIR)

from data import download_tiny_shakespeare, get_batch, load_text_data
from model import GPT, GPTConfig
from train import TrainConfig, load_checkpoint, resolve_device, run_experiment, set_seed

## 数据

字符级 tokenizer 的 vocabulary 来自语料中所有不同字符。输入 `x` 与目标 `y` 相差一个字符。

In [ ]:
data_path = download_tiny_shakespeare(STAGE_DIR / 'data' / 'tinyshakespeare.txt')
text_data = load_text_data(data_path)
device = resolve_device('auto')
print('device:', device)
print('vocab size:', text_data.tokenizer.vocab_size)
print('train tokens:', len(text_data.train), 'val tokens:', len(text_data.val))
print('characters:', ''.join(text_data.tokenizer.chars))

## 诊断：固定 batch 过拟合

预测：如果模型、loss 和反向传播链路正确，重复训练同一个小 batch 时 loss 应快速下降。这个实验不衡量泛化。

In [ ]:
set_seed(42)
diagnostic_config = GPTConfig(
    vocab_size=text_data.tokenizer.vocab_size,
    block_size=32,
    n_layer=2,
    n_head=2,
    n_embd=64,
)
diagnostic_model = GPT(diagnostic_config).to(device)
optimizer = torch.optim.AdamW(diagnostic_model.parameters(), lr=3e-3)
x_fixed, y_fixed = get_batch(
    text_data.train, batch_size=8, block_size=32, device=device
)
diagnostic_losses = []
for step in range(201):
    _, loss = diagnostic_model(x_fixed, y_fixed)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    diagnostic_losses.append(loss.item())

print(f'loss: {diagnostic_losses[0]:.4f} → {diagnostic_losses[-1]:.4f}')
plt.plot(diagnostic_losses)
plt.xlabel('step')
plt.ylabel('fixed-batch loss')
plt.title('Can the model overfit one batch?')
plt.show()

## Baseline

第一次先不调参。运行前记录预测：train loss 与 val loss 会同时下降；后期 train loss 通常下降得更快。

如需把 checkpoint 放到持久目录，可在运行前设置环境变量 `NANOGPT_RUN_DIR`。

In [ ]:
config_values = json.loads(
    (STAGE_DIR / 'configs' / 'tiny_shakespeare_baseline.json').read_text()
)
config_values['data_path'] = str(STAGE_DIR / 'data' / 'tinyshakespeare.txt')
config_values['run_dir'] = os.environ.get(
    'NANOGPT_RUN_DIR', str(STAGE_DIR / 'runs' / 'baseline')
)
baseline_config = TrainConfig(**config_values)
baseline_config

In [ ]:
result = run_experiment(baseline_config)

In [ ]:
history = result['history']
steps = [row['step'] for row in history]
plt.plot(steps, [row['train_loss'] for row in history], label='train')
plt.plot(steps, [row['val_loss'] for row in history], label='val')
plt.xlabel('step')
plt.ylabel('cross-entropy loss')
plt.legend()
plt.show()

## 固定 prompt 采样

In [ ]:
model = result['model']
tokenizer = result['data'].tokenizer
prompt = 'ROMEO:'
prompt_ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)
set_seed(42)
model.eval()
generated = model.generate(prompt_ids, 400, temperature=0.8, top_k=20)
print(tokenizer.decode(generated[0].tolist()))

## Checkpoint 恢复验证

不仅确认文件存在，还重新构造模型与 optimizer，并比较相同输入下的 logits。

In [ ]:
checkpoint_path = Path(result['checkpoint'])
restored_model, checkpoint, restored_optimizer = load_checkpoint(checkpoint_path, device)
restored_model.eval()
with torch.no_grad():
    original_logits, _ = model(prompt_ids)
    restored_logits, _ = restored_model(prompt_ids)
torch.testing.assert_close(original_logits, restored_logits)
print('checkpoint restored:', checkpoint_path)
print('saved step:', checkpoint['step'])
print('optimizer state entries:', len(restored_optimizer.state))

## 实验后记录

- 固定 batch loss 是否明显下降？
- baseline 的最低 val loss 是多少？
- train/val gap 从什么时候出现？
- 生成文本先学会了哪些结构？
- GPU 型号、训练时间和最大显存是多少？
- 下一次实验只改变哪个变量？